# 31 · CXR-LT — trích embedding RAD-DINO (Stage 3a)

Chạy `microsoft/rad-dino` **nguyên bản, đông cứng** một lượt trên 377,110 ảnh và lưu embedding.
Không fine-tune, không head tuỳ biến — đây đúng nghĩa *"RAD-DINO mặc định"*, và nó lập **sàn** cho
mọi thứ về sau.

Sau khi cache xong, mọi baseline ở `32` (CXR-only, EHR+CXR, so ba arm) đều thành bài toán bảng
chạy trên CPU trong vài phút.

## Ba arm ảnh — chạy **từng arm một**

| `ARM` | nguồn | biến đổi lúc nạp | model input | GPU |
|---|---|---|---|---|
| `crop518` | `cxr_560_shards/` + `cxr_560_shards_cxrlt/` | `CenterCrop(518)` từ 559 | 518 | ~30 ph |
| `lb518` | `cxr_1022_letterbox_cxrlt/` | resize 1022 → 518, **không crop** | 518 | ~30 ph |
| `lb1022` | `cxr_1022_letterbox_cxrlt/` | **không đổi gì** | 1022 | **~3.3 h** |

Ba arm cho phép tách hai hiệu ứng thay vì chỉ thấy tổng:

```
crop518 -> lb518   : FOV 71.5% -> 100%,  do phan giai 4.1% -> 2.9% pixel goc
lb518   -> lb1022  : FOV giu nguyen 100%, do phan giai 2.9% -> 11.2%   <- CO LAP do phan giai
```

`lb1022` chạy ở kích thước **khác** lúc RAD-DINO được huấn luyện (518), nên positional embedding bị
nội suy 37×37 → 73×73. `Dinov2Embeddings` tự làm, không cần cờ. Nhưng **đó là lệch phân bố** —
nếu `lb1022` thua `lb518` dù độ phân giải cao hơn thì thủ phạm là nội suy, và có `lb518` mới phân
biệt được.

## Lưu cả CLS lẫn trung bình patch

`REPORT_frozen_token_fusion_revised.md` từng đo patch hơn CLS trên cohort cũ. Đó là **giả thuyết
cần kiểm lại từ đầu** ở đây, không mang số cũ sang — nhưng cache cả hai thì phép kiểm gần như miễn
phí, nên không có lý do gì để chỉ lưu một.

**GPU.** Có `RESUME` theo shard — đứt thì chạy lại, bỏ qua phần đã xong. Cần `28`, `28b`, `28c`.

## C0 · Config — chọn `ARM`

In [1]:
import os, io, json, time, tarfile
import numpy as np, pandas as pd
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

# Chay bao nhieu arm tuy y trong MOT lan — model giong nhau ca ba, chi khac kich
# thuoc dau vao, nen nap RAD-DINO mot lan roi lap. Arm da xong bi bo qua nho _done_<arm>.json.
ARMS = ['crop518', 'lb518', 'nosq518']    # crop518/lb518 da xong -> chay lai chi ton vai giay
#ARMS = ['crop518', 'lb518', 'nosq518', 'lb1022']

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings'); os.makedirs(EMB_DIR, exist_ok=True)
P_IDX   = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
assert os.path.exists(P_IDX), f'thieu {P_IDX} -> chay 28 truoc'

_CROP = [(os.path.join(DATA_DIR, 'cxr_560_shards'),       'shard_'),
         (os.path.join(DATA_DIR, 'cxr_560_shards_cxrlt'), 'cxrlt_')]
_LB   = [(os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt'), 'lb_')]
# arm -> (nguon shard, kich thuoc CANH DAI dua vao model, che do tien xu ly)
#   crop   : 559 -> CenterCrop(518)            FOV 71.5%, dem 0
#   square : letterbox 1022 -> resize NxN      FOV 100%,  dem ~18%
#   nosq   : cat dai den -> giu ti le, dai=N   FOV 100%,  dem 0        <<< arm moi
ARM_CFG = {
    'crop518': (_CROP,  518, 'crop'),
    'lb518':   (_LB,    518, 'square'),
    'lb1022':  (_LB,   1022, 'square'),
    'nosq518': (_LB,    518, 'nosq'),
}
assert set(ARMS) <= set(ARM_CFG), f'ARM la: {sorted(ARM_CFG)}'

BATCH = 64               # VRAM khong phai rang buoc; tang neu GPU khoe
AMP   = True             # fp16 autocast
BOX_THR = 20             # nguong pixel coi la den khi tim khung noi dung (JPEG co ringing)

print(f'se chay {len(ARMS)} arm: {ARMS}')
for a in ARMS:
    src, tg, mode = ARM_CFG[a]
    for d, _ in src:
        assert os.path.isdir(d), f'arm {a}: khong thay {d} -> chay 28b/28c truoc'
    print(f'  {a:<8} canh dai {tg:>4}px | che do {mode:<6} | '
          + ' + '.join(os.path.basename(d) for d, _ in src))
print(f'''
nosq518 la phep CO LAP cua rieng bien DEM:
  lb518   canh dai 518, FOV 100%, dem ~18%
  nosq518 canh dai 518, FOV 100%, dem   0%     <- chi khac dung mot thu
Do phan giai va FOV giu nguyen, nen chenh lech giua hai arm nay CHI do dai den.''')

Mounted at /content/drive
se chay 3 arm: ['crop518', 'lb518', 'nosq518']
  crop518  canh dai  518px | che do crop   | cxr_560_shards + cxr_560_shards_cxrlt
  lb518    canh dai  518px | che do square | cxr_1022_letterbox_cxrlt
  nosq518  canh dai  518px | che do nosq   | cxr_1022_letterbox_cxrlt

nosq518 la phep CO LAP cua rieng bien DEM:
  lb518   canh dai 518, FOV 100%, dem ~18%
  nosq518 canh dai 518, FOV 100%, dem   0%     <- chi khac dung mot thu
Do phan giai va FOV giu nguyen, nen chenh lech giua hai arm nay CHI do dai den.


## C1 · Nạp RAD-DINO + lấy chuẩn hoá **từ chính processor**

Không hard-code `image_mean`/`image_std` — đọc từ processor để nếu HuggingFace đổi thì ta biết,
chứ không âm thầm chuẩn hoá sai.

Cell cũng in cảnh báo khi `TARGET != 518`, kèm số patch và chi phí attention thực tế.

In [2]:
import torch
from transformers import AutoModel, AutoImageProcessor

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG co GPU -> doi runtime sang GPU truoc khi chay'
print(f'GPU: {torch.cuda.get_device_name(0)} | '
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

proc  = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
model = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
for p in model.parameters(): p.requires_grad_(False)

MEAN = torch.tensor(proc.image_mean, device=dev).view(1, 3, 1, 1)
STD  = torch.tensor(proc.image_std,  device=dev).view(1, 3, 1, 1)
PATCH      = model.config.patch_size
MODEL_SIZE = model.config.image_size
D          = model.config.hidden_size
print(f'\nRAD-DINO: patch {PATCH} | huan luyen o {MODEL_SIZE}px | hidden {D}')
print(f'chuan hoa tu processor: mean {proc.image_mean} std {proc.image_std}')

_g0 = MODEL_SIZE // PATCH
print()
for a in ARMS:
    tg = ARM_CFG[a][1]
    assert tg % PATCH == 0, f'{a}: {tg} khong chia het cho patch {PATCH}'
    _g = tg // PATCH
    line = f'  {a:<8} {tg:>4}px -> {_g}x{_g} = {_g*_g:>6,} patch'
    if tg != MODEL_SIZE:
        line += (f'  !! pos-embed NOI SUY {_g0}x{_g0} -> {_g}x{_g} '
                 f'(x{_g/_g0:.2f}), attention ~{(_g*_g/(_g0*_g0))**2:.0f}x')
    print(line)
if any(ARM_CFG[a][1] != MODEL_SIZE for a in ARMS):
    print('\n   Noi suy pos-embed la LECH PHAN BO so voi luc huan luyen — phai DO,')
    print('   khong duoc gia dinh la tot. Do la ly do arm lb518 ton tai.')

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | 102 GB


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


RAD-DINO: patch 14 | huan luyen o 518px | hidden 768
chuan hoa tu processor: mean (0.5307, 0.5307, 0.5307) std (0.2583, 0.2583, 0.2583)

  crop518   518px -> 37x37 =  1,369 patch
  lb518     518px -> 37x37 =  1,369 patch
  nosq518   518px -> 37x37 =  1,369 patch


## C2 · Thứ tự chuẩn + bản đồ shard

Một **thứ tự hàng duy nhất** dùng chung cho cả ba arm, để `32` xếp chồng embedding mà không phải
join lại. Thứ tự đó là `sort_values(['subject_id','study_id','dicom_id'])` — trùng cách `28c` xếp
shard, nên vừa đọc vừa ghi đều tuần tự.

In [3]:
IDX = pd.read_parquet(P_IDX, columns=['dicom_id', 'study_id', 'subject_id',
                                      'split', 'view_group'])
ORDER = (IDX.sort_values(['subject_id', 'study_id', 'dicom_id'])
            .reset_index(drop=True))
ROW = {d: i for i, d in enumerate(ORDER.dicom_id)}       # dicom_id -> hang trong .npy
N = len(ORDER)

print(f'{N:,} anh | thu tu: subject_id, study_id, dicom_id')
for a in ARMS:                      # so tar cua tung arm (TARS dung trong vong lap o C4)
    _n = sum(1 for d, pref in ARM_CFG[a][0] for f in os.listdir(d)
             if f.endswith('.tar') and f.startswith(pref))
    print(f'  {a:<8} {_n:>4} tar')

P_ORD = os.path.join(EMB_DIR, 'emb_index.parquet')
if os.path.exists(P_ORD):
    _o = pd.read_parquet(P_ORD)
    assert (_o.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all(), \
        'emb_index.parquet da ton tai voi THU TU KHAC -> khong duoc tron hai lan chay'
    print('emb_index.parquet: thu tu khop ban da co')
else:
    ORDER.to_parquet(P_ORD, index=False)
    print(f'ghi {P_ORD}')

377,110 anh | thu tu: subject_id, study_id, dicom_id
  crop518   378 tar
  lb518     378 tar
  nosq518   378 tar
emb_index.parquet: thu tu khop ban da co


## C3 · Hàm nạp ảnh — **khác nhau đúng một chỗ giữa các arm**

```
crop518 : shard 559px -> CenterCrop(518)          (tai lap dung 04a/16)
lb518   : shard 1022px -> resize 518, KHONG crop  (crop se xen lai phan vien vua giu duoc)
lb1022  : shard 1022px -> giu nguyen
```

Ảnh lưu ở chế độ `L` (một kênh); RAD-DINO nhận 3 kênh nên lặp lại kênh — đúng cách `04b`/`16` làm.

In [4]:
def _decode(b):
    im = Image.open(io.BytesIO(b))
    return im if im.mode == 'L' else im.convert('L')

def _content_box(im, thr=BOX_THR):
    '''Khung bao vung KHAC den. Voi anh letterbox, day chinh la anh goc truoc khi dem.
       Tra None neu anh toan den (khong co gi de cat).'''
    a = np.asarray(im)
    r, c = a.max(1) > thr, a.max(0) > thr
    if not r.any() or not c.any(): return None
    y0, y1 = int(r.argmax()), len(r) - int(r[::-1].argmax())
    x0, x1 = int(c.argmax()), len(c) - int(c[::-1].argmax())
    return x0, y0, x1, y1

def load_batch(blobs):
    '''Che do crop/square: bytes JPEG -> tensor [B,3,TARGET,TARGET] da chuan hoa, tren GPU.'''
    arr = np.empty((len(blobs), TARGET, TARGET), dtype='uint8')
    for i, b in enumerate(blobs):
        im = _decode(b); w, h = im.size
        if MODE == 'crop':                             # 559 -> CenterCrop(518)
            assert w >= TARGET and h >= TARGET, f'anh {w}x{h} nho hon {TARGET}, khong crop duoc'
            l, t = (w - TARGET) // 2, (h - TARGET) // 2
            im = im.crop((l, t, l + TARGET, t + TARGET))
        elif (w, h) != (TARGET, TARGET):               # 1022 -> 518 (arm lb*)
            im = im.resize((TARGET, TARGET), Image.BICUBIC)
        arr[i] = np.asarray(im, dtype='uint8')
    return to_tensor(arr)

def prep_nosq(b):
    '''Che do nosq -> (mang uint8 [H,W], khoa gom nhom (H,W)).

       Canh dai = TARGET; canh ngan lam tron ve BOI SO cua PATCH bang `round`.
       Dung `round` + gian nhe (<= 7/430 = 1.6% mot truc) chu khong `ceil` + dem:
       muc tieu cua arm nay la dem = 0 TUYET DOI, nen khong duoc de lai du 1.6%
       — neu ton hai cua dem la dang BAO HOA thi 1.6% con lai van co the du gay ra no.

       Chi cat den o truc NGAN la dai letterbox; phan collimation dinh mep truc DAI
       (do duoc ~2.4% o 33 C2) cung bi cat theo — do la khac biet phu duy nhat so voi
       lb518, va no chi doi do phan giai giai phau ~2.4%.'''
    im = _decode(b)
    bx = _content_box(im)
    if bx is not None: im = im.crop(bx)
    w, h = im.size
    _k = lambda v: max(PATCH, min(TARGET, int(round(v / PATCH)) * PATCH))
    if h >= w: H, W = TARGET, _k(TARGET * w / h)
    else:      H, W = _k(TARGET * h / w), TARGET
    return np.asarray(im.resize((W, H), Image.BICUBIC), dtype='uint8'), (H, W)

def to_tensor(arr):
    '''[B,H,W] uint8 -> [B,3,H,W] float da chuan hoa, tren GPU.'''
    x = torch.from_numpy(arr).to(dev).unsqueeze(1).float().div_(255)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

@torch.no_grad()
def embed(x):
    '''-> (cls [B,D], patch-mean [B,D]). Bo CLS khi lay trung binh patch.'''
    with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
        h = model(pixel_values=x).last_hidden_state    # [B, 1+P, D]
    return h[:, 0].float(), h[:, 1:].mean(1).float()

@torch.no_grad()
def embed_blobs(blobs):
    '''Dung cho cac cell KIEM TRA: chay duoc ca ba che do. Che do nosq xu ly tung anh
       mot vi kich thuoc khac nhau khong xep chung batch duoc — cham nhung dung.'''
    if MODE != 'nosq':
        return embed(load_batch(blobs))
    cs, ps = [], []
    for b in blobs:
        arr, _ = prep_nosq(b)
        c, p = embed(to_tensor(arr[None])); cs.append(c); ps.append(p)
    return torch.cat(cs), torch.cat(ps)

# --- kiem tung arm tren mot anh that truoc khi chay hang gio ---
for a in ARMS:
    src, TARGET, MODE = ARM_CFG[a]             # dat global cho load_batch/prep_nosq doc
    _tar = next(os.path.join(d, f) for d, pref in src
                for f in sorted(os.listdir(d)) if f.endswith('.tar') and f.startswith(pref))
    with tarfile.open(_tar) as t:
        _m = [m for m in t.getmembers() if m.name.endswith('.jpg')][:4]
        _b = [t.extractfile(m).read() for m in _m]
        _sz = Image.open(io.BytesIO(_b[0])).size
    _c, _p = embed_blobs(_b)
    assert _c.shape[1] == D and torch.isfinite(_c).all() and torch.isfinite(_p).all()
    _extra = ''
    if MODE == 'nosq':
        _shp = []
        for _bb in _b:
            _a4, _key = prep_nosq(_bb); _shp.append(_key)
            assert _key[0] % PATCH == 0 and _key[1] % PATCH == 0, f'{_key} khong chia het {PATCH}'
            assert (_a4.max(1) > BOX_THR).all() and (_a4.max(0) > BOX_THR).all(), \
                'con hang/cot toan den sau khi cat -> prep_nosq chua bo het dem'
        _extra = f' | hinh dang {_shp} | dem 0.00%'
    print(f'  {a:<8} shard {_sz[0]}px -> canh dai {TARGET}px | cls |mean| {_c.abs().mean():.3f} '
          f'| patch |mean| {_p.abs().mean():.3f}{_extra}')

  crop518  shard 559px -> canh dai 518px | cls |mean| 0.338 | patch |mean| 0.164
  lb518    shard 1022px -> canh dai 518px | cls |mean| 0.336 | patch |mean| 0.237
  nosq518  shard 1022px -> canh dai 518px | cls |mean| 0.339 | patch |mean| 0.173 | hinh dang [(518, 490), (518, 420), (518, 420), (518, 420)] | dem 0.00%


/tmp/ipykernel_9203/3052503936.py:51: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(arr).to(dev).unsqueeze(1).float().div_(255)


## C4 · Trích — memmap + `RESUME` theo shard

Ghi thẳng vào `.npy` dạng memmap thay vì giữ hết trong RAM (377,110 × 768 × 4 byte × 2 = 2.3 GB —
vừa RAM, nhưng memmap thì đứt giữa chừng không mất gì).

Danh sách shard đã xong nằm trong `_done_<arm>.json`, nên chạy lại là bỏ qua đúng phần đã làm.

In [5]:
from tqdm.auto import tqdm

def _mm(path):
    if os.path.exists(path):
        return np.lib.format.open_memmap(path, mode='r+')
    return np.lib.format.open_memmap(path, mode='w+', dtype='float32', shape=(N, D))

def paths(arm):
    return (os.path.join(EMB_DIR, f'emb_{arm}_cls.npy'),
            os.path.join(EMB_DIR, f'emb_{arm}_patch.npy'),
            os.path.join(EMB_DIR, f'_done_{arm}.json'))

def tars_of(arm):
    return [os.path.join(d, f) for d, pref in ARM_CFG[arm][0]
            for f in sorted(os.listdir(d)) if f.endswith('.tar') and f.startswith(pref)]

T_ALL = time.time()
for arm in ARMS:
    src, TARGET, MODE = ARM_CFG[arm]           # global cho load_batch/prep_nosq doc
    P_CLS, P_PAT, P_DONE = paths(arm)
    TARS = tars_of(arm)
    CLS, PAT = _mm(P_CLS), _mm(P_PAT)
    done = set(json.load(open(P_DONE))) if os.path.exists(P_DONE) else set()
    todo = [t for t in TARS if os.path.basename(t) not in done]
    print(f'\n=== {arm} (canh dai {TARGET}px, che do {MODE}) — '
          f'{len(done)}/{len(TARS)} tar da xong, con {len(todo)} ===')
    if not todo:
        print('   bo qua, da trich xong'); CLS.flush(); PAT.flush(); continue

    n_img, n_shape, t0 = 0, set(), time.time()
    pbar = tqdm(total=len(todo), unit='tar', desc=f'embed {arm}', dynamic_ncols=True)
    try:
        for tp in todo:
            with tarfile.open(tp) as t:
                ms = [m for m in t.getmembers() if m.name.endswith('.jpg')]
                if MODE == 'nosq':
                    # Kich thuoc khac nhau khong xep chung batch duoc -> GOM theo (H,W).
                    # Canh ngan da lam tron ve boi so 14 nen chi co ~30 khoa, moi khoa
                    # van du anh de batch day. Giai ma ca tar truoc: ~998 x 518 x 430
                    # uint8 ~ 220 MB, chap nhan duoc.
                    grp = {}
                    for m in ms:
                        a_, key = prep_nosq(t.extractfile(m).read())
                        grp.setdefault(key, []).append((ROW[m.name[:-4]], a_))
                    n_shape |= set(grp)
                    for key, items in grp.items():
                        for i in range(0, len(items), BATCH):
                            ch = items[i:i+BATCH]
                            c, p = embed(to_tensor(np.stack([a_ for _, a_ in ch])))
                            rows = [r for r, _ in ch]
                            CLS[rows] = c.cpu().numpy(); PAT[rows] = p.cpu().numpy()
                            n_img += len(ch)
                    del grp
                else:
                    for i in range(0, len(ms), BATCH):
                        chunk = ms[i:i+BATCH]
                        rows = [ROW[m.name[:-4]] for m in chunk]
                        c, p = embed(load_batch([t.extractfile(m).read() for m in chunk]))
                        CLS[rows] = c.cpu().numpy(); PAT[rows] = p.cpu().numpy()
                        n_img += len(chunk)
            # FLUSH TRUOC roi moi ghi _done. Nguoc lai thi session chet giua chung se de lai
            # _done khai "da xong" trong khi du lieu con nam o page cache, chua xuong Drive
            # -> hang toan 0 ma resume khong bao gio trich lai. Da dinh dung loi nay hai lan.
            CLS.flush(); PAT.flush()
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(P_DONE, 'w'))
            pbar.update(1)
            pbar.set_postfix_str(f'{n_img:,} anh | {n_img/max(time.time()-t0,1e-9):.0f} img/s'
                                 + (f' | {len(n_shape)} hinh dang' if MODE == 'nosq' else ''))
    finally:
        pbar.close(); CLS.flush(); PAT.flush()
    _el = time.time() - t0
    print(f'   {n_img:,} anh trong {_el/60:.1f} phut ({n_img/max(_el,1e-9):.0f} img/s) | '
          f'{os.path.getsize(P_CLS)/1e6:.0f} MB x2')
    # Doc LAI tu dia (khong dung memmap dang mo) de chac du lieu da thuc su xuong Drive.
    _chk = np.load(P_CLS, mmap_mode='r')
    _z = int((np.abs(_chk[::37]).sum(1) == 0).sum())
    assert _z == 0, (f'{arm}: {_z} hang (trong mau) van toan 0 ngay sau khi trich -> '
                     f'ghi KHONG xuong dia. Xoa _done_{arm}.json roi chay lai.')
    del _chk
    if MODE == 'nosq':
        _g = sorted(n_shape)
        print(f'   {len(_g)} hinh dang dau vao, tu {_g[0]} den {_g[-1]} — dem 0% o moi hinh dang')
print(f'\nTONG {len(ARMS)} arm: {(time.time()-T_ALL)/60:.1f} phut')


=== crop518 (canh dai 518px, che do crop) — 378/378 tar da xong, con 0 ===
   bo qua, da trich xong

=== lb518 (canh dai 518px, che do square) — 378/378 tar da xong, con 0 ===
   bo qua, da trich xong

=== nosq518 (canh dai 518px, che do nosq) — 378/378 tar da xong, con 0 ===
   bo qua, da trich xong

TONG 3 arm: 0.1 phut


## C5 · Cổng kiểm chứng

Ba thứ phải đúng trước khi `32` tin vào file này:

1. **không hàng nào bị bỏ sót** — hàng toàn 0 nghĩa là ảnh đó chưa được trích
2. **tái lập được** — trích lại một batch cố định phải cho cùng kết quả `max|Δ| < 1e-3`
   (không phải 0 tuyệt đối: `fp16` autocast không tất định bit-for-bit)
3. **không suy biến** — CLS và patch-mean phải khác nhau, và phương sai giữa các ảnh phải > 0

In [6]:
EMB = {}
for arm in ARMS:
    src, TARGET, MODE = ARM_CFG[arm]
    P_CLS, P_PAT, P_DONE = paths(arm); TARS = tars_of(arm)
    done = set(json.load(open(P_DONE))) if os.path.exists(P_DONE) else set()
    assert len(done) == len(TARS), f'{arm}: moi xong {len(done)}/{len(TARS)} tar -> chay lai C4'

    CLS = np.load(P_CLS, mmap_mode='r'); PAT = np.load(P_PAT, mmap_mode='r')
    EMB[arm] = (CLS, PAT)
    _zero = (np.abs(CLS[::37]).sum(1) == 0).sum()        # lay mau moi 37 hang
    assert _zero == 0, f'{arm}: {_zero} hang (trong mau) toan 0 -> co anh chua trich'

    # tai lap: nguong 1e-3 chu khong phai 0 — fp16 autocast khong tat dinh bit-for-bit
    with tarfile.open(TARS[0]) as t:
        ms = [m for m in t.getmembers() if m.name.endswith('.jpg')][:16]
        b  = [t.extractfile(m).read() for m in ms]
        rows = [ROW[m.name[:-4]] for m in ms]
    # fp16 autocast KHONG tat dinh bit-for-bit, va cuBLAS doi kernel theo KICH THUOC BATCH:
    # C4 chay batch 64, cell nay chay batch 16 — rieng nosq* la batch 1 (moi anh mot hinh dang)
    # nen dung han GEMV thay GEMM -> lech ~1.5e-3. Do la NHIEU, khong phai loi.
    # Cham bang sai so TUONG DOI + cosine: lech hang / nham anh lam cosine sup do,
    # con nhieu fp16 thi khong dong den cosine. Nguong tuyet doi cu la sai tu goc vi no
    # phu thuoc bien do embedding.
    c2, p2 = embed_blobs(b); c2, p2 = c2.cpu().numpy(), p2.cpu().numpy()
    _rc, _rp = CLS[rows], PAT[rows]
    _rel = lambda a, r: float(np.abs(a - r).max() / (np.abs(r).max() + 1e-9))
    _cs  = lambda a, r: float(((a*r).sum(1) /
                (np.linalg.norm(a, axis=1)*np.linalg.norm(r, axis=1) + 1e-9)).min())
    d_c, d_p = _rel(c2, _rc), _rel(p2, _rp)
    k = min(_cs(c2, _rc), _cs(p2, _rp))
    assert k > 0.9995, (f'{arm}: trich lai ra KHAC (cosine {k:.5f}) -> nghi LECH HANG hoac '
                        f'nham anh, day khong phai nhieu fp16')
    assert max(d_c, d_p) < 5e-3, f'{arm}: sai so tuong doi {max(d_c,d_p):.2e} qua lon'

    _s, _t = CLS[::37], PAT[::37]
    assert _s.std(0).mean() > 1e-3 and _t.std(0).mean() > 1e-3, f'{arm}: embedding gan hang so'
    _cos = ((_s*_t).sum(1) / (np.linalg.norm(_s,axis=1)*np.linalg.norm(_t,axis=1)+1e-9)).mean()
    print(f'  {arm:<8} khong bo sot | tai lap rel {max(d_c,d_p):.1e} cos {k:.6f} | '
          f'std cls {_s.std(0).mean():.4f} patch {_t.std(0).mean():.4f} | '
          f'cos(cls,patch) {_cos:.3f}')

# --- cac arm PHAI khac nhau; neu giong thi mot arm khong co tac dung ---
if len(ARMS) > 1:
    print()
    for i in range(len(ARMS)):
        for j in range(i+1, len(ARMS)):
            a, b_ = ARMS[i], ARMS[j]
            d = np.abs(EMB[a][1][::37] - EMB[b_][1][::37]).max()
            assert d > 1e-4, f'{a} va {b_} ra embedding GIONG NHAU -> mot arm khong co tac dung'
            print(f'  {a} vs {b_}: max|d| patch {d:.3f}  (khac nhau, dung nhu mong doi)')

_miss = [a for a in ARM_CFG if a not in ARMS]
print(f'''
================================================================
XONG {len(ARMS)} ARM: {ARMS}

  emb_<arm>_cls.npy / emb_<arm>_patch.npy   moi cai [{N:,}, {D}]
  emb_index.parquet                          thu tu hang, DUNG CHUNG moi arm

CON THIEU: {_miss if _miss else 'khong'}
  Them vao ARMS o C0 roi chay lai tu C0 — arm da xong duoc bo qua nho _done_<arm>.json.

DOC KET QUA nosq518 o 32 (33 §11):
  cls  phuc hoi lai phan da mat  (+0.0098 head, +0.0327 tail so voi lb518)
       -> DAI DEN la thu pham, khoi phai chay lb1022.
  cls  van thap nhu lb518
       -> khong phai dai den; lb1022 (2.3-3.3h) moi tra loi duoc.
  patch giu duoc muc loi o tail cua lb518 (+0.0194) trong ca hai kich ban.

TIEP THEO -> 32_cxrlt_baselines.ipynb (them 'nosq518' vao ARMS cua no)
================================================================''')

  crop518  khong bo sot | tai lap rel 1.8e-07 cos 1.000000 | std cls 0.4420 patch 0.1052 | cos(cls,patch) 0.154
  lb518    khong bo sot | tai lap rel 1.4e-07 cos 1.000000 | std cls 0.4429 patch 0.1206 | cos(cls,patch) 0.100
  nosq518  khong bo sot | tai lap rel 5.6e-04 cos 0.999999 | std cls 0.4446 patch 0.1016 | cos(cls,patch) 0.120

  crop518 vs lb518: max|d| patch 2.336  (khac nhau, dung nhu mong doi)
  crop518 vs nosq518: max|d| patch 1.184  (khac nhau, dung nhu mong doi)
  lb518 vs nosq518: max|d| patch 1.758  (khac nhau, dung nhu mong doi)

XONG 3 ARM: ['crop518', 'lb518', 'nosq518']

  emb_<arm>_cls.npy / emb_<arm>_patch.npy   moi cai [377,110, 768]
  emb_index.parquet                          thu tu hang, DUNG CHUNG moi arm

CON THIEU: ['lb1022']
  Them vao ARMS o C0 roi chay lai tu C0 — arm da xong duoc bo qua nho _done_<arm>.json.

DOC KET QUA nosq518 o 32 (33 §11):
  cls  phuc hoi lai phan da mat  (+0.0098 head, +0.0327 tail so voi lb518)
       -> DAI DEN la thu pham, khoi 